# Survival / Reliability Analysis Practice Notebook

This notebook is a hands-on companion to the Markdown file on **Survival / Reliability Analysis**.  
It demonstrates practical failure-time and reliability methods used in engineering, maintenance, and risk modeling.

Topics covered:

1. Survival function basics  
2. Kaplan–Meier estimator  
3. Weibull analysis  
4. Hazard modeling  
5. Failure rate analysis  
6. Reliability estimation  
7. Series and parallel system reliability  
8. Cox proportional hazards intuition  

The notebook is designed for learning, GitHub repositories, and classroom use.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import weibull_min
from scipy.optimize import curve_fit

np.random.seed(42)

## Create Example Failure-Time Data

We generate synthetic failure and censoring data.  
In reliability studies, some items fail during observation, while others are still alive when observation ends.

In [ ]:
n = 120

# True failure times from a Weibull-like process
true_failure_times = weibull_min.rvs(c=1.8, scale=900, size=n, random_state=42)

# Random censoring times
censoring_times = np.random.uniform(400, 1400, size=n)

# Observed time and event indicator
observed_time = np.minimum(true_failure_times, censoring_times)
event_observed = (true_failure_times <= censoring_times).astype(int)

surv_df = pd.DataFrame({
    'time': observed_time,
    'event': event_observed
})
surv_df.head()

## 1. Survival Function Basics

The survival function is:

$$
S(t) = P(T > t)
$$

For complete uncensored data, a simple empirical estimate at time \(t\) is the fraction of items surviving beyond \(t\).

In [ ]:
time_grid = np.linspace(0, surv_df['time'].max(), 100)
empirical_survival = [(surv_df['time'] > t).mean() for t in time_grid]

plt.figure(figsize=(8, 4))
plt.plot(time_grid, empirical_survival)
plt.title('Empirical Survival Fraction')
plt.xlabel('Time')
plt.ylabel('Survival Probability')
plt.show()

## 2. Kaplan–Meier Estimator

The Kaplan–Meier estimator is:

$$
\hat{S}(t)=\prod_{t_i \le t}\left(1-\frac{d_i}{n_i}\right)
$$

where:
- \(d_i\) = number of failures at time \(t_i\)
- \(n_i\) = number at risk just before time \(t_i\)


In [ ]:
def kaplan_meier(times, events):
    df = pd.DataFrame({'time': times, 'event': events}).sort_values('time')
    event_times = np.sort(df.loc[df['event'] == 1, 'time'].unique())
    survival = []
    s = 1.0
    for t in event_times:
        n_i = (df['time'] >= t).sum()
        d_i = ((df['time'] == t) & (df['event'] == 1)).sum()
        s *= (1 - d_i / n_i)
        survival.append((t, s, n_i, d_i))
    return pd.DataFrame(survival, columns=['time', 'survival', 'n_at_risk', 'events'])

km_df = kaplan_meier(surv_df['time'].values, surv_df['event'].values)
km_df.head()

In [ ]:
plt.figure(figsize=(8, 4))
plt.step(km_df['time'], km_df['survival'], where='post')
plt.title('Kaplan–Meier Survival Curve')
plt.xlabel('Time')
plt.ylabel('Estimated Survival')
plt.show()

## 3. Weibull Analysis

The Weibull survival model is:

$$
S(t)=\exp\left[-\left(\frac{t}{\eta}\right)^\beta\right]
$$

The hazard function is:

$$
h(t)=\frac{\beta}{\eta}\left(\frac{t}{\eta}\right)^{\beta-1}
$$

Here we fit a Weibull distribution to the observed failure times only.

In [ ]:
fail_times = surv_df.loc[surv_df['event'] == 1, 'time'].values
shape_hat, loc_hat, scale_hat = weibull_min.fit(fail_times, floc=0)

pd.DataFrame({
    'Parameter': ['shape_beta', 'scale_eta'],
    'Estimate': [shape_hat, scale_hat]
})

In [ ]:
t_plot = np.linspace(1, surv_df['time'].max(), 200)
weibull_survival = np.exp(- (t_plot / scale_hat) ** shape_hat)

plt.figure(figsize=(8, 4))
plt.step(km_df['time'], km_df['survival'], where='post', label='Kaplan–Meier')
plt.plot(t_plot, weibull_survival, label='Weibull Fit')
plt.title('Kaplan–Meier vs Weibull Survival Fit')
plt.xlabel('Time')
plt.ylabel('Survival Probability')
plt.legend()
plt.show()

## 4. Hazard Modeling

The hazard function is:

$$
h(t)=\frac{f(t)}{S(t)}
$$

For the fitted Weibull model, we can evaluate hazard across time.

In [ ]:
weibull_hazard = (shape_hat / scale_hat) * (t_plot / scale_hat) ** (shape_hat - 1)

plt.figure(figsize=(8, 4))
plt.plot(t_plot, weibull_hazard)
plt.title('Weibull Hazard Function')
plt.xlabel('Time')
plt.ylabel('Hazard')
plt.show()

### Cumulative Hazard

For a Weibull model:

$$
H(t)=\left(\frac{t}{\eta}\right)^\beta
$$

In [ ]:
cum_hazard = (t_plot / scale_hat) ** shape_hat

plt.figure(figsize=(8, 4))
plt.plot(t_plot, cum_hazard)
plt.title('Cumulative Hazard Function')
plt.xlabel('Time')
plt.ylabel('Cumulative Hazard')
plt.show()

## 5. Failure Rate Analysis

A simple failure rate estimate is:

$$
\lambda = \frac{\text{number of failures}}{\text{total observed operating time}}
$$

In [ ]:
num_failures = surv_df['event'].sum()
total_time = surv_df['time'].sum()
failure_rate = num_failures / total_time
mtbf = 1 / failure_rate

pd.DataFrame({
    'Measure': ['Number of failures', 'Total observed time', 'Failure rate', 'MTBF'],
    'Value': [num_failures, total_time, failure_rate, mtbf]
})

## 6. Reliability Estimation

Reliability is the same as the survival probability:

$$
R(t)=S(t)
$$

For exponential failure:

$$
R(t)=e^{-\lambda t}
$$

In [ ]:
exp_reliability = np.exp(-failure_rate * t_plot)

plt.figure(figsize=(8, 4))
plt.plot(t_plot, exp_reliability)
plt.title('Exponential Reliability Curve')
plt.xlabel('Time')
plt.ylabel('Reliability')
plt.show()

### Mean Time To Failure (MTTF)

For the exponential model:

$$
MTTF = \frac{1}{\lambda}
$$

In [ ]:
mttf = 1 / failure_rate
mttf

## 7. Series and Parallel System Reliability

For components with reliabilities \(R_1, R_2, \dots, R_n\):

### Series system
$$
R_s = \prod_{i=1}^{n} R_i
$$

### Parallel system
$$
R_p = 1 - \prod_{i=1}^{n}(1-R_i)
$$

In [ ]:
component_reliabilities = np.array([0.96, 0.93, 0.97, 0.95])
series_reliability = np.prod(component_reliabilities)
parallel_reliability = 1 - np.prod(1 - component_reliabilities)

pd.DataFrame({
    'System Type': ['Series', 'Parallel'],
    'Reliability': [series_reliability, parallel_reliability]
})

## 8. Cox Proportional Hazards Intuition

The Cox model is:

$$
h(t|X)=h_0(t)\exp(\beta^T X)
$$

Here we show the idea of a **hazard ratio** without fitting a full Cox package.

If a covariate coefficient is \(\beta\), then the hazard ratio for a one-unit increase is:

$$
HR=e^{\beta}
$$

In [ ]:
beta_example = 0.7
hazard_ratio = np.exp(beta_example)

pd.DataFrame({
    'Quantity': ['beta', 'Hazard Ratio exp(beta)'],
    'Value': [beta_example, hazard_ratio]
})

Interpretation:

- if hazard ratio > 1, risk increases  
- if hazard ratio < 1, risk decreases  
- if hazard ratio = 1, no change in hazard

## 9. Small Summary Table

This table gathers the main outputs from the notebook.

In [ ]:
summary = pd.DataFrame({
    'Measure': [
        'Kaplan–Meier final survival',
        'Weibull shape beta',
        'Weibull scale eta',
        'Failure rate',
        'MTBF',
        'Series system reliability',
        'Parallel system reliability',
        'Example hazard ratio'
    ],
    'Value': [
        km_df['survival'].iloc[-1],
        shape_hat,
        scale_hat,
        failure_rate,
        mtbf,
        series_reliability,
        parallel_reliability,
        hazard_ratio
    ]
})
summary

## 10. Mini Exercises

Try these on your own:

1. Change the Weibull shape parameter in the synthetic data generation and inspect the hazard curve.  
2. Increase censoring and see how the Kaplan–Meier curve changes.  
3. Compare exponential and Weibull reliability curves.  
4. Create a five-component series and parallel system.  
5. Replace the synthetic data with real fatigue-life or maintenance data.  
6. Extend the Cox hazard-ratio example using real covariates and a dedicated survival package.

These exercises are especially useful for structural reliability, predictive maintenance, fatigue modeling, and digital-twin applications.